# Week 2: three-corpus theme validation

Executed October 2, 2026. This notebook runs the pipeline on the richer fictional 1,000-comment benchmark, a fixed 300-narrative CFPB sample, and 300 FAA comments obtained through the public Mirrulations archive of Regulations.gov. The original short/repetitive synthetic fixture is also retained as a stress test.

Public narratives and review packets stay local. Saved outputs contain aggregate results only. An executed notebook is not a human review or a guarantee of theme quality. Formal gate approvals are no longer a prerequisite, as reported by the fellow on October 1; quality checks remain applicable.

The richer fictional fixture is newly authored development data, not a replacement claim about the old benchmark. Both fixture results are shown. Timing excludes imports and output serialization.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import torch
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "pipeline").is_dir())
sys.path.insert(0, str(ROOT))
from scripts.validate_cfpb_themes import load_sample
from scripts.validate_core import load_federal
from scripts.validate_week2 import run
torch.set_num_threads(2)
OUTPUT = ROOT / "data/results/week2-notebook"
OUTPUT.mkdir(parents=True, exist_ok=True)
corpora = {
    "synthetic": pd.read_csv(ROOT / "data/samples/synthetic_theme_benchmark.csv"),
    "cfpb": load_sample(ROOT / "data/public_samples/cfpb_2017_300")[0],
    "federal": load_federal(ROOT / "data/public_samples/federal_mirror_300")[0],
    "synthetic_legacy_stress": pd.read_csv(ROOT / "data/samples/synthetic_chicago_hearing.csv"),
}
summary = {}

## Synthetic

In [2]:
summary["synthetic"] = run(corpora["synthetic"], "synthetic", OUTPUT)
record = summary["synthetic"]
display(pd.DataFrame([{k: record[k] for k in ["rows", "theme_count", "outlier_share", "seconds", "quote_counts", "excerpt_count"]}]))
display(pd.DataFrame(list(record["checks"].items()), columns=["Check", "Passed"]))

/Users/vamshikrishna/Desktop/Hearing-Lens-Work-Handoff-2026-09-16/Hearing-Lens-working/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17496.08it/s]

,rows,theme_count,outlier_share,seconds,quote_counts,excerpt_count
0,1000,6,0.044,11.556035,"[3, 3, 3, 3, 3, 3]",4


,Check,Passed
0,five_to_fifteen_themes,True
1,outliers_under_twenty_percent,True
2,runtime_under_two_minutes,True
3,three_quotes_each,True
4,valid_source_quotes,True
5,distinct_quotes,True
6,all_assignments_preserved,True
7,membership_totals,True


## Cfpb

In [3]:
summary["cfpb"] = run(corpora["cfpb"], "cfpb", OUTPUT)
record = summary["cfpb"]
display(pd.DataFrame([{k: record[k] for k in ["rows", "theme_count", "outlier_share", "seconds", "quote_counts", "excerpt_count"]}]))
display(pd.DataFrame(list(record["checks"].items()), columns=["Check", "Passed"]))

,rows,theme_count,outlier_share,seconds,quote_counts,excerpt_count
0,300,6,0.166667,11.144434,"[3, 3, 3, 3, 3, 3]",18


,Check,Passed
0,five_to_fifteen_themes,True
1,outliers_under_twenty_percent,True
2,runtime_under_two_minutes,True
3,three_quotes_each,True
4,valid_source_quotes,True
5,distinct_quotes,True
6,all_assignments_preserved,True
7,membership_totals,True


## Federal

In [4]:
summary["federal"] = run(corpora["federal"], "federal", OUTPUT)
record = summary["federal"]
display(pd.DataFrame([{k: record[k] for k in ["rows", "theme_count", "outlier_share", "seconds", "quote_counts", "excerpt_count"]}]))
display(pd.DataFrame(list(record["checks"].items()), columns=["Check", "Passed"]))

,rows,theme_count,outlier_share,seconds,quote_counts,excerpt_count
0,300,6,0.19,7.423241,"[3, 3, 3, 3, 3, 3]",17


,Check,Passed
0,five_to_fifteen_themes,True
1,outliers_under_twenty_percent,True
2,runtime_under_two_minutes,True
3,three_quotes_each,True
4,valid_source_quotes,True
5,distinct_quotes,True
6,all_assignments_preserved,True
7,membership_totals,True


## Synthetic Legacy Stress

In [5]:
summary["synthetic_legacy_stress"] = run(corpora["synthetic_legacy_stress"], "synthetic_legacy_stress", OUTPUT)
record = summary["synthetic_legacy_stress"]
display(pd.DataFrame([{k: record[k] for k in ["rows", "theme_count", "outlier_share", "seconds", "quote_counts", "excerpt_count"]}]))
display(pd.DataFrame(list(record["checks"].items()), columns=["Check", "Passed"]))

,rows,theme_count,outlier_share,seconds,quote_counts,excerpt_count
0,1000,6,0.133,1.388131,"[3, 2, 2, 2, 2, 2]",3


,Check,Passed
0,five_to_fifteen_themes,True
1,outliers_under_twenty_percent,True
2,runtime_under_two_minutes,True
3,three_quotes_each,False
4,valid_source_quotes,True
5,distinct_quotes,True
6,all_assignments_preserved,True
7,membership_totals,True


In [6]:
(OUTPUT / "week2-summary.json").write_text(json.dumps(summary, indent=2))
print("All four runs completed. Detailed redacted review packets are local only.")

All four runs completed. Detailed redacted review packets are local only.


## Qualitative review and limits

See `docs/week2_validation_2026-10-02.md` for the actual review findings and limitations. The local packets include three largest themes per corpus with a fixed systematic selection of members plus every selected quote. Reviewers must inspect original context for excerpts. Only contiguous complete sentences are eligible; no comment is padded or paraphrased.

Week 3 human sentiment/emotion labeling is a separate task. No human labels or approval results are manufactured by this notebook. Public source samples remain ignored by Git; preparation instructions and checksums are published.